# Representational Geometry as a Fidelity Metric for Connectome-Constrained Neural Emulations
**Experiment 4: Untrained Networks | Michael Zhou | Georgia Tech ECE**

> **Correction notice.** An earlier version of this experiment reported that untrained
> connectome-constrained networks show directional geometry before training (CC vs
> Rand-syn r = 0.260, p_perm = 0.041; CC vs Rand-sign r = 0.215, p_perm = 0.048), and
> concluded that wiring encodes a geometric prior that training amplifies. **Those
> statistics are numerical noise.** The untrained mean CC RDM has an off-diagonal span of
> 1.66×10⁻⁸ against a float32 round-off floor of 1.93×10⁻⁷ — the floor is eleven times
> the signal (span/floor = 0.09×). `scipy.spatial.distance.cosine` preserves input dtype;
> with float32 population vectors of magnitude ≈ 0.5, the quotient in `1 − dot/(‖a‖‖b‖)`
> sits within one ULP of 1.0 and the subtraction retains no significant digits. A
> cancellation-free cross-check (Euclidean on L2-normalized vectors) gives per-model
> Kendall τ = 1.0000 against cosine for all 150 models: both metrics see the same thing,
> and what they see is rounding. That the two headline p-values landed just under 0.05 is
> what random rank vectors do.
>
> This notebook now computes RDMs in float64 and **aborts at a precision guard** rather
> than reporting statistics on unresolvable matrices. On the data as configured, the guard
> fires.

Experiments 1–2 established that pretrained connectome-constrained (CC) networks produce
representational geometries distinguishable from weight-shuffled random baselines, and
that CC networks carry within-polarity circular direction structure the random baselines
lack (ON-ON r = 0.937 vs 0.38, against an explicit circular reference). Those networks are
both connectome-constrained **and** task-trained on optic flow, leaving a confound: does
the geometry arise from the wiring, or from training?

This experiment attempts to isolate the wiring contribution by comparing three conditions
of *untrained* networks — networks at the Flyvis prior initialization, before any
gradient-based optimization:

- **Untrained CC:** `Network()` with default Flyvis architecture, connectome fixed, free
  parameters (`nodes_bias`, `nodes_time_const`, `edges_syn_strength`) perturbed with
  Gaussian noise to generate an ensemble of N distinct seeds. No checkpoint loaded.
  `edges_sign` and `edges_syn_count` are fixed by the connectome and never change.
- **Untrained Random (syn shuffle):** Same untrained CC networks, with
  `edges_syn_strength` shuffled in a sign-preserving manner after perturbation. Absolute
  values shuffled within each sign class, preserving E/I identity. Matches the Shiu-style
  baseline design from Experiments 1–3.
- **Untrained Random (sign shuffle):** Deeper disruption — `edges_sign` also randomly
  reassigned across all 604 cell-type pairs, preserving the total E/I count but scrambling
  which pairs are excitatory vs inhibitory. Tests whether E/I wiring identity contributes
  to representational geometry before training.

**The question, and the prior question.** The intended comparison was: does connectome
wiring produce geometrically distinct population codes compared to shuffled baselines
*before* task training? That question presupposes a measurable geometry. It does not have
one. Untrained networks at the Flyvis prior produce population responses so nearly
identical across the twelve directions that their pairwise cosine distances (~10⁻⁸) fall
below the numerical resolution of the float32 responses they derive from. **The prior
question — is the RDM resolvable at all? — must be answered first, and at this
perturbation scale the answer is no.**

**What that means.** This is a result, not a failure. If untrained connectome-constrained
networks have no measurable directional geometry, then training does not *amplify* a
wiring prior — training *creates* the directional structure. That inverts the original
conclusion and is a stronger claim, because it is falsifiable: it predicts that geometry
should emerge at some perturbation scale, or not at all. The perturbation-sensitivity
sweep (`exp4_perturbation_sweep.ipynb`) tests exactly this, with the shuffled conditions
as the control that distinguishes a wiring prior from the trivial effect of large
perturbations producing large responses.

**Stimuli:** 12 ON moving edge directions (0°–330°, 30° increments), speed=19, matching
Experiment 1 exactly.

**Metrics:** float64 cosine RDM (chord form), Euclidean-normalized RDM as a
cancellation-free cross-check, precision guard on RDM resolvability, circularity control
against an explicit circular-distance reference, RSA (Spearman r, Kendall τ_A) with both
analytical and permutation p-values, permutation test (Nili et al. 2014, 10,000
permutations).

**The biological reference is not used here.** The von Mises T4/T5 reference is degenerate
on the ON-only stimulus set — T5 does not respond to ON edges (Maisak et al. Fig. 3c/3d)
and T5a–d duplicate T4a–d exactly, so the effective reference is four cardinal von Mises
curves of identical width, correlating with a pure angular-distance matrix at r = 0.978
(see Experiment 3). The "inversion" reported earlier — untrained CC at r = −0.015 against
biology while shuffled baselines reached 0.471 and 0.585 — is the same confound: those
RDMs' correlations with the circular reference are −0.046, +0.427, and +0.600, tracking
their biological correlations to within 0.05. The circularity control replaces this
comparison.

**Runtime:** Google Colab T4 GPU. Roughly 40 s per model (13 forward passes: 1 stability
check + 12 stimuli), so N=50 × 3 conditions ≈ **100 minutes**.

**References:**
- Lappalainen et al. 2024. Connectome-constrained networks predict neural activity across the fly visual system. *Nature* 634, 1132–1140. https://www.nature.com/articles/s41586-024-07939-3
- Maisak et al. 2013. A directional tuning map of Drosophila elementary motion detectors. *Nature* 500, 212–216. https://doi.org/10.1038/nature12320
- Shiu et al. 2024. A Drosophila computational brain model reveals sensorimotor processing. *Nature* 634, 210–219. https://www.nature.com/articles/s41586-024-07763-9
- Kriegeskorte et al. 2008. Representational similarity analysis. *Frontiers in Systems Neuroscience* 2:4. https://www.frontiersin.org/journals/systems-neuroscience/articles/10.3389/neuro.06.004.2008/full
- Kriegeskorte & Wei 2021. Neural tuning and representational geometry. *Nature Reviews Neuroscience* 22, 703–718. https://www.nature.com/articles/s41583-021-00502-3
- Nili et al. 2014. A toolbox for representational similarity analysis. *PLOS Computational Biology* 10(4): e1003553. https://doi.org/10.1371/journal.pcbi.1003553
- Brunton et al. 2026. The digital sphinx: Can a worm brain control a fly body? *bioRxiv*. https://www.biorxiv.org/content/10.64898/2026.03.20.713233v1

## Setup

Install Flyvis and download pretrained models. Select **Runtime → Change runtime type → GPU**
before running. Tested on T4 GPU (14.56 GiB).

In [ ]:
# Install packages
!git clone https://github.com/TuragaLab/flyvis.git
%cd /content/flyvis
!pip install -e .[examples]
!flyvis download-pretrained

Cloning into 'flyvis'...
remote: Enumerating objects: 5230, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 5230 (delta 170), reused 148 (delta 148), pack-reused 5028 (from 3)
Receiving objects: 100% (5230/5230), 188.38 MiB | 14.22 MiB/s, done.
Resolving deltas: 100% (2679/2679), done.
/content/flyvis
Obtaining file:///content/flyvis
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 43.5 MB/s e

## Experiment

**Stimuli:** 12 ON moving edge conditions — 12 directions at 30° increments (0° through
330°), speed=19, matching Experiment 1 exactly. ON polarity only (intensity=1).

**Networks:**
- *Untrained connectome-constrained (CC):* N=50 models constructed via `Network()` at the
  Flyvis prior initialization — `nodes_bias ~ Normal(mean=0.5, std=0.05, seed=0)`,
  `nodes_time_const = 0.05` (constant), and `edges_syn_strength = 0.01 / mean_syn_count`
  (see the clamp note below). No checkpoint loaded. Free parameters (`nodes_bias`,
  `nodes_time_const`, `edges_syn_strength`) perturbed with Gaussian noise per seed
  (σ = 0.05, 0.005, 0.002 respectively) to generate an ensemble of N distinct models.
  `edges_sign` and `edges_syn_count` are fixed by the connectome and unchanged throughout.
- *Untrained Random (syn shuffle):* Same untrained CC networks with `edges_syn_strength`
  shuffled in a sign-preserving manner (absolute values shuffled within each sign class),
  preserving E/I identity. Matches the Shiu-style baseline design from Experiments 1–3.
- *Untrained Random (sign shuffle):* Deeper disruption — `edges_sign` randomly reassigned
  across all 604 cell-type pairs, preserving the total E/I count but scrambling which
  pairs are excitatory vs inhibitory. Tests whether E/I wiring identity contributes to
  geometry before training.

**Stability constraint:** Candidate networks are rejected and resampled if any activation
exceeds ±1×10⁶ or is non-finite on a single forward-pass check stimulus. Up to
MAX_ATTEMPTS=100 seeds tried per accepted model. The check does **not** wrap the forward
pass in `try/except`: a CUDA error or shape mismatch is not dynamic instability, and the
original implementation would have counted it as such, silently. Measured: **0 rejections,
50/50 first-try acceptance in all three conditions.** (The original reported "mean attempts
25.5 ± 14.4," which was the mean of the cumulative seed counter 1…50 — a loop index, not a
resampling statistic. The acceptance claim was true; the number cited to support it was
meaningless.)

**Population vectors:** Peak central-cell voltage per cell type (65-dim) in response to
each stimulus condition, extracted via `LayerActivity.central[ct].squeeze().numpy().max()`,
**cast to float64 immediately** and accumulated in float64 thereafter.

**Note on the synapse-strength clamp — it silences the densest connections.**
`edges_syn_strength` (class `SynapseCountScaling`, `scale = 0.01`,
`groupby = [source_type, target_type]`, `clamp = non_negative`) is **one scaling factor per
cell-type pair — 604 in total** — initialized as `0.01 / mean_syn_count`, where
`mean_syn_count` is the mean synapses per column-to-column instance of that pair. The
scaling is **inverse** to synapse count (verified: Pearson r = 1.0000, max residual
2.4×10⁻⁷), so factors span 6.9×10⁻⁵ (144 synapses/instance) to 3.7×10⁻² (0.27), and the
*densest* connections carry the *smallest* factors.

Consequently a σ = 0.002 perturbation drives the densest pairs below zero first, and the
clamp silences them. Across 50 seeds this zeroes **8.1% of the 604 cell-type pairs (range
5.6–10.3%)**, strongly biased by density: silencing probability falls monotonically from
**40.7% in the lowest-factor decile (11–144 synapses/instance) to 0% in the top four
deciles (0.27–1.5)**; median `mean_syn_count` of silenced pairs is 11.4 against a
population median of 1.99 (Mann–Whitney p = 3×10⁻²²). Untrained CC networks are therefore
**sparsified by removal of their highest-synapse-count cell-type connections** — an entire
cell-type-to-cell-type projection deleted, not a few synapses attenuated — rather than
merely perturbed.

Task training also zeroes factors (31/604 in the reference model, so the clamp is an active
mechanism rather than a dormant safety rail), but selects a nearly disjoint set (overlap 3;
2.5 expected under independence) that is **not** detectably density-biased (median 2.8;
Mann–Whitney p = 0.080, limited power at n = 31). The density bias belongs to the untrained
perturbation protocol, not to Flyvis. The sweep (Experiment 4b) shows the bias is a
small-σ phenomenon: by σ = 0.128 every decile is silenced at ~48% and the fraction reaches
48.2%.

**Reproducibility:** seed=42 (`numpy`, `torch`, `torch.cuda`);
`torch.use_deterministic_algorithms(True)`.

**Metrics:**
- **Cosine distance RDM (float64, chord form).** Computed as
  `1 − cos θ = ‖â − b̂‖² / 2`, a sum of squares that cannot cancel. The float64 cast is the
  substantive fix: `scipy.spatial.distance.cosine` preserves input dtype, and float32
  inputs destroy distances at the 10⁻⁸ scale these networks produce.
- **Euclidean-normalized RDM.** Euclidean distance between L2-normalized vectors. Monotone
  in the angle, no subtraction from 1, therefore cancellation-free. Reported as a
  cross-check: per-model Kendall τ against the cosine RDM is the direct test of whether a
  rank order is real (τ ≈ 1 ⇒ metric is irrelevant ⇒ signal; τ ≈ 0 ⇒ arbitrary ⇒ noise).
- **Precision guard.** The RDM's off-diagonal span is compared against the float32
  round-off floor implied by the response magnitude (`eps_f32 × max|response|`). If the
  span does not exceed the floor by a comfortable margin, the notebook **aborts**. No
  Spearman, Kendall, or permutation statistic is computed on an unresolvable RDM. The
  absence of this check is what allowed the original Experiment 4 to report permutation
  tests on rounding. The margin (10×) is a heuristic, not a derived threshold — trust the
  verdict when lopsided, not when marginal; the cross-check above does not depend on it.
- **Circularity control.** Each mean RDM is correlated against the circular-distance
  reference min(|i−j|, 12−|i−j|). This asks directly whether a network orders directions
  by angle, against an explicit reference rather than a biological proxy.
- Spearman r and Kendall's τ_A RDM correlation, with **both analytical and permutation
  p-values** reported for every comparison. (The original reported only p_perm = 0.048 for
  CC vs Rand-sign, silently dropping the analytical p = 0.084.)
- Stimulus-label permutation test — nonparametric inference on RDM correlations (Nili et
  al. 2014, 10,000 permutations), with +1 smoothing.
- Within-CC consistency — stability of untrained CC representational geometry across seeds.

**The biological reference is computed but not interpreted as a fidelity measure.** The
von Mises T4/T5 reference is degenerate on the ON-only stimulus set: T5 does not respond
to ON edges (Maisak et al. Fig. 3c/3d) and T5a–d duplicate T4a–d exactly, so the effective
reference is four cardinal von Mises curves of identical width, correlating with a pure
angular-distance matrix at r = 0.978 (see Experiment 3). It is retained in the output and
the saved `.npz` only so the confound can be reproduced. The "inversion" reported earlier —
untrained CC at r = −0.015 against biology while shuffled baselines reached 0.471 and
0.585 — is exactly this: those RDMs' correlations with the circular reference are −0.046,
+0.427, and +0.600. The **circularity control** is the interpretable quantity.

**Runtime:** ~40 s per model (13 forward passes: 1 stability check + 12 stimuli), so
N=50 × 3 conditions ≈ 100 minutes on a Colab T4.

In [ ]:
"""
Experiment 4 (CORRECTED): Untrained Connectome-Constrained Networks

WHAT WAS WRONG WITH THE ORIGINAL
--------------------------------
The original reported RDM values "on the order of 1e-8 to 1e-7" and argued that
"what matters is the relative geometry within each RDM, not the absolute scale."
At that scale, computed the way the original computed them, there is no relative
geometry to speak of.

The population matrices are float32. `scipy.spatial.distance.cosine` preserves
the input dtype: given a float32 array it forms dot products and norms in
float32 and evaluates 1 - dot/(|a||b|). For the near-identical responses that
untrained networks produce across the twelve directions, that quotient lies
within one float32 ULP (~1.19e-07) of 1.0, so the subtraction retains no
significant digits.

Measured directly, on synthetic data matching the real regime (65-dim vectors,
|values| ~0.5, direction-to-direction differences ~1e-4):

    float32 input : RDM off-diagonal range 0.0 .. 1.79e-07   (contains exact zeros)
    float64 input : RDM off-diagonal range 8.1e-09 .. 1.66e-08
    max |f32 - f64| = 1.69e-07

The error is an order of magnitude larger than the entire dynamic range of the
true RDM. Rank order -- the only thing Spearman and Kendall see -- is therefore
set by rounding. On the same synthetic data, r(RDM, circular reference) reads
-0.073 from the float32 path and -0.161 from the float64 path.

On the real saved data the same instability appears, and worse. Recomputing
`cc_mean_rdm` through a float64 path moves r(RDM, circular) from -0.046 to
+0.463. Two different float64 recomputation paths give +0.175 and +0.463; they
disagree with each other because there is no correct value to recover -- the
float32 responses never resolved the distances. The stored CC RDM spans
4.05e-08 against a float32 round-off floor of 1.93e-07: the floor is five times
the span. The two shuffled conditions have larger responses, larger spans, and
shift by only 0.003 under the same recomputation. So the reported "progressive
degradation" from CC through syn-shuffle to sign-shuffle compares one noise
matrix against two signal matrices.

Consequently the Experiment 4 statistics -- r = 0.260 and r = 0.215 with
p_perm = 0.041 and 0.048, the biological "inversion" at -0.015 / 0.471 / 0.585,
and within-CC consistency 0.006 +/- 0.133 -- are permutation tests on noise.
That the two headline p-values landed just under 0.05, as random rank vectors
do, is consistent with this.

Two things that are NOT the cause, checked and excluded:

  * The `+1e-10` epsilon in `build_rdm`. In float32, 0.53 + 1e-10 == 0.53
    exactly (the ULP at 0.53 is ~6e-08), so the epsilon vanished. Applied in
    float64 it changes the RDM by ~5e-16. It is inert either way -- neither
    the problem nor a mitigation.
  * Catastrophic cancellation in the algebraic form of cosine distance itself.
    Given float64 inputs, `scipy.cosine` is well conditioned in this regime and
    agrees with the chord identity to six significant figures. The dtype of the
    input, not the formula, is what destroys the result.

WHAT THIS VERSION CHANGES
-------------------------
1. Population vectors are accumulated in float64 from the start.
2. Cosine distance is computed by `cosine_distance_stable`, which casts to
   float64 unconditionally -- that cast is the substantive fix -- and evaluates
   the chord identity 1 - cos(theta) = ||a_hat - b_hat||^2 / 2. The chord form
   is a sum of squares and costs nothing, but it is not what repairs the result:
   scipy's own formula is well conditioned given float64 inputs. The dtype is
   the fix.
3. A MANDATORY PRECISION GUARD (`assert_rdm_resolvable`) compares the RDM's
   dynamic range against the round-off floor implied by the response precision.
   If the RDM is not resolvable, the script ABORTS rather than reporting numbers.
   This is the check whose absence produced the original error.
4. The inert `+1e-10` epsilon is removed (zero-norm vectors are handled
   explicitly instead).
5. `attempts_log` is fixed. The original appended the cumulative seed counter,
   so with zero rejections it recorded [1, 2, ..., 50] and reported
   "mean attempts 25.5 +/- 14.4" -- which is just mean/SD of 1..50 and carries
   no information. It now counts real rejections per accepted model.
6. Both analytical and permutation p-values are reported for every comparison.
   The original silently dropped the analytical p for CC vs Rand-sign
   (p = 0.084, not significant) and reported only p_perm = 0.048.
7. A circularity control is added: every RDM is correlated against the
   circular-distance reference min(|i-j|, 12-|i-j|). The biological von Mises
   reference is itself ~0.978 circular, so a raw correlation against it does not
   demonstrate biological fidelity; the partial correlation controlling for
   circular structure is reported instead.

WHAT THIS VERSION CANNOT FIX
----------------------------
If untrained network responses are genuinely near-identical across the twelve
directions -- and the ~1e-8 cosine distances suggest they are -- then no
numerical fix creates signal that is not there. The precision guard will abort,
and the honest conclusion is that untrained networks have no measurable
representational geometry at this perturbation scale. That is itself a finding:
it would mean training CREATES the directional geometry rather than amplifying a
pre-existing wiring prior, which is the opposite of the original claim.

Two remedies, if the guard fires:
  (a) Increase perturbation magnitude (BIAS_NOISE etc.) until responses separate
      across directions, and report the sensitivity. The original used
      near-default noise, which may be why responses barely differ.
  (b) Use a metric that does not cancel: Euclidean distance on L2-normalized
      population vectors carries the same angular information without the
      1 - cos(theta) subtraction. It is computed alongside cosine, and the
      Kendall tau between the two metrics' rank orders is reported before the
      guard. Disagreement between them is direct evidence of noise and does not
      rely on the RESOLVE_MARGIN heuristic.

Neither is a cosmetic change; both alter what the experiment measures, and the
choice should be made deliberately rather than silently.
"""

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from scipy.stats import spearmanr, kendalltau, rankdata

# ── REPRODUCIBILITY ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_MODELS       = 50
N_PERMUTATIONS = 10_000
MAX_ATTEMPTS   = 100
OVERFLOW_LIMIT = 1e6

BIAS_NOISE         = 0.05
TIME_CONST_NOISE   = 0.005
SYN_STRENGTH_NOISE = 0.002

# Precision guard. The network responses are produced in float32; the round-off
# in any quantity derived from them is ~eps_f32 * |response|. If the RDM's
# off-diagonal range does not exceed this by a comfortable margin, the RDM's
# rank order is noise.
FLOAT32_EPS   = float(np.finfo(np.float32).eps)   # 1.19e-07
# RESOLVE_MARGIN is a heuristic, not a derived threshold. There is no principled
# basis for 10x; it was chosen for headroom. The floor is crude in both
# directions: np.abs(pop_matrix).max() takes the largest response across ALL
# models and stimuli, so one outlier inflates the floor for the whole ensemble,
# while averaging 50 RDMs suppresses noise by ~sqrt(50), which the floor does
# not credit. Treat the verdict as informative only when it is lopsided (well
# under 1x, or well over 100x). A marginal ratio -- 8x, 12x -- means the guard
# cannot decide, not that the RDM is borderline-acceptable. The cosine-vs-
# euclidean cross-check below is the direct test and does not depend on this
# constant.
RESOLVE_MARGIN = 10.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

results_dir = Path("/content/results"); results_dir.mkdir(parents=True, exist_ok=True)
figures_dir = Path("/content/figures"); figures_dir.mkdir(parents=True, exist_ok=True)

import flyvis
from flyvis.datasets.moving_bar import MovingEdge
from flyvis.network import Network
from flyvis.utils.activity_utils import LayerActivity

ANGLES    = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]
INTENSITY = 1
N_DIR     = len(ANGLES)


# ── NUMERICALLY STABLE DISTANCE ───────────────────────────────────────────────
def cosine_distance_stable(a, b):
    """Cosine distance via the chord identity, evaluated in float64.

        1 - cos(theta) = ||a_hat - b_hat||^2 / 2

    Two properties matter here. First, inputs are cast to float64 unconditionally,
    which is the substantive fix: `scipy.spatial.distance.cosine` preserves the
    input dtype, and float32 inputs destroy distances at the 1e-8 scale these
    networks produce. Second, the right-hand side is a sum of squares and so
    never cancels, which costs nothing and removes one way to get this wrong.

    Agrees with scipy.cosine to full precision on float64 inputs.
    """
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0.0 or nb == 0.0:
        return np.nan          # undefined; caller must handle
    d = a / na - b / nb
    return float(d @ d) / 2.0


def euclidean_normalized(a, b):
    """Euclidean distance between L2-normalized vectors.

    Monotone in the angle, no subtraction from 1, so no cancellation. Reported
    as a cancellation-free cross-check on the cosine RDM.
    """
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0.0 or nb == 0.0:
        return np.nan
    return float(np.linalg.norm(a / na - b / nb))


def build_rdm(pop_matrix, metric="cosine"):
    """RDM from an (n_stim, n_cells) population matrix, computed in float64.

    No epsilon is added. In the original code `pop_matrix + 1e-10` was applied to
    float32 values of magnitude ~0.5, where 1e-10 is far below the ULP (~6e-8)
    and therefore vanished entirely. Zero-norm rows are handled explicitly.
    """
    P = np.asarray(pop_matrix, dtype=np.float64)
    if not np.isfinite(P).all():
        raise ValueError("non-finite population vector reached build_rdm")

    n = P.shape[0]
    fn = cosine_distance_stable if metric == "cosine" else euclidean_normalized
    R = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            d = fn(P[i], P[j])
            if np.isnan(d):
                raise ValueError(f"zero-norm population vector at stimulus {i} or {j}")
            R[i, j] = R[j, i] = d
    return R


# ── THE GUARD THAT WAS MISSING ────────────────────────────────────────────────
def rdm_resolution_report(rdm, pop_matrix, label):
    """Is this RDM above the numerical noise floor implied by its inputs?

    Even computed in float64, an RDM whose entries are comparable to the
    round-off of the float32 responses they derive from carries no reliable rank
    order. The caller MUST NOT report correlations computed on a non-resolvable
    RDM. The absence of this check is what allowed the original Experiment 4 to
    report permutation tests on noise.

    Returns (resolvable: bool, report: str).
    """
    iu = np.triu_indices(rdm.shape[0], k=1)
    off = rdm[iu]
    rng_ = off.max() - off.min()

    # Round-off inherited from float32 responses of this magnitude.
    resp_scale = float(np.abs(pop_matrix).max())
    floor = FLOAT32_EPS * resp_scale

    resolvable = rng_ > RESOLVE_MARGIN * floor
    lines = [
        f"  [{label}] RDM off-diagonal range : {off.min():.3e} .. {off.max():.3e}"
        f"  (span {rng_:.3e})",
        f"  [{label}] response |max|          : {resp_scale:.3e}",
        f"  [{label}] float32 round-off floor : {floor:.3e}"
        f"  (= eps_f32 x |response|)",
        f"  [{label}] span / floor            : {rng_ / floor:.2f}x"
        f"   (need >= {RESOLVE_MARGIN:.0f}x)",
        f"  [{label}] RESOLVABLE              : {resolvable}",
    ]
    return resolvable, "\n".join(lines)


def assert_rdm_resolvable(rdm, pop_matrix, label):
    ok, report = rdm_resolution_report(rdm, pop_matrix, label)
    print(report)
    if not ok:
        raise SystemExit(
            f"\nABORT: the {label} RDM's dynamic range does not exceed the "
            f"float32 round-off floor of the responses it was computed from.\n"
            f"Its rank order -- and therefore every Spearman/Kendall statistic "
            f"and permutation test derived from it -- would be numerical noise.\n\n"
            f"This is exactly the failure mode of the original Experiment 4, "
            f"whose RDM entries (7e-9 .. 5e-8) sat below a ~6e-8 floor.\n\n"
            f"Remedies (each changes what the experiment measures; choose "
            f"deliberately):\n"
            f"  (a) raise BIAS_NOISE / SYN_STRENGTH_NOISE until untrained "
            f"responses actually differ across directions, and report the "
            f"sensitivity analysis;\n"
            f"  (b) use metric='euclidean_normalized', which is cancellation-"
            f"free;\n"
            f"  (c) conclude that untrained networks have no measurable "
            f"representational geometry at this perturbation scale -- which "
            f"would mean training CREATES the directional structure rather than "
            f"amplifying a wiring prior.\n"
        )


# ── STATS ─────────────────────────────────────────────────────────────────────
def upper(M):
    return M[np.triu_indices_from(M, k=1)]


def circular_reference(n=N_DIR):
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    d = np.abs(i - j)
    return np.minimum(d, n - d).astype(float)


def rdm_similarity(r1, r2):
    """Spearman and Kendall with BOTH analytical p-values returned.

    The original reported only the permutation p for CC vs Rand-sign, silently
    dropping the analytical p = 0.084 (not significant).
    """
    a, b = upper(r1), upper(r2)
    rs, ps = spearmanr(a, b)
    rk, pk = kendalltau(a, b)
    return rs, ps, rk, pk


def permutation_test_rdm(r1, r2, n_perm=N_PERMUTATIONS, seed=SEED):
    rng = np.random.default_rng(seed)
    n = r1.shape[0]
    a = upper(r1)
    obs_r  = spearmanr(a, upper(r2)).statistic
    obs_t  = kendalltau(a, upper(r2)).statistic
    null_r = np.empty(n_perm); null_t = np.empty(n_perm)
    for k in range(n_perm):
        p = rng.permutation(n)
        b = upper(r2[np.ix_(p, p)])
        null_r[k] = spearmanr(a, b).statistic
        null_t[k] = kendalltau(a, b).statistic
    # +1 smoothing: an unsmoothed p can report exactly 0.0, which is not a
    # valid estimate from a finite permutation sample.
    p_r = (np.sum(null_r >= obs_r) + 1) / (n_perm + 1)
    p_t = (np.sum(null_t >= obs_t) + 1) / (n_perm + 1)
    return obs_r, p_r, obs_t, p_t, null_r, null_t


def partial_spearman(a, b, c):
    """Spearman partial correlation of a,b controlling for c."""
    ra, rb, rc = rankdata(a), rankdata(b), rankdata(c)
    C = np.column_stack([np.ones_like(rc), rc])
    resa = ra - C @ np.linalg.lstsq(C, ra, rcond=None)[0]
    resb = rb - C @ np.linalg.lstsq(C, rb, rcond=None)[0]
    den = np.linalg.norm(resa) * np.linalg.norm(resb)
    return np.nan if den == 0 else float(resa @ resb / den)


# ── SIMULATION (float64 accumulation) ─────────────────────────────────────────
def get_cell_types(network):
    return [ct.decode() if isinstance(ct, bytes) else ct
            for ct in network.connectome.unique_cell_types[:]]


def get_population_vector(network, stimulus, dt, cell_types):
    """Peak central-cell voltage per cell type, returned as float64.

    The network computes in float32; we cast once, immediately, and do all
    downstream arithmetic in float64. This does not recover precision the
    network never had -- see the resolution guard -- but it prevents adding
    further loss.
    """
    if stimulus.dim() == 2:
        stimulus = stimulus.unsqueeze(1)
    initial_state = network.fade_in_state(1.0, dt, stimulus[[0]])
    with torch.no_grad():
        responses = network.simulate(stimulus[None], dt,
                                     initial_state=initial_state).cpu()
    layer_act = LayerActivity(responses, network.connectome, keepref=True)
    pop_vec = np.array(
        [layer_act.central[ct].squeeze().numpy().max() for ct in cell_types],
        dtype=np.float64,
    )
    pop_vec = np.clip(pop_vec, -OVERFLOW_LIMIT, OVERFLOW_LIMIT)
    del responses, layer_act
    torch.cuda.empty_cache()
    return pop_vec


# ── NETWORK CONSTRUCTION (unchanged from original) ────────────────────────────
def make_untrained_cc(seed):
    rng = np.random.default_rng(seed)
    net = Network()
    with torch.no_grad():
        net.nodes_bias.data += torch.tensor(
            rng.normal(0, BIAS_NOISE, size=net.nodes_bias.shape), dtype=torch.float32)
        net.nodes_time_const.data = torch.clamp(
            net.nodes_time_const.data + torch.tensor(
                rng.normal(0, TIME_CONST_NOISE, size=net.nodes_time_const.shape),
                dtype=torch.float32), min=0.001)
        raw = net.edge_params.syn_strength.raw_values
        raw.data = torch.clamp(raw.data + torch.tensor(
            rng.normal(0, SYN_STRENGTH_NOISE, size=raw.shape),
            dtype=torch.float32), min=0.0)
    return net.to(DEVICE)


def apply_syn_shuffle(net, rng):
    with torch.no_grad():
        raw = net.edge_params.syn_strength.raw_values
        v = raw.data.cpu().numpy().copy()
        pos, neg = np.where(v > 0)[0], np.where(v <= 0)[0]
        if len(pos) > 1: v[pos] = rng.permutation(v[pos])
        if len(neg) > 1: v[neg] = rng.permutation(v[neg])
        raw.data = torch.tensor(v, dtype=torch.float32).to(DEVICE)
    return net


def apply_sign_shuffle(net, rng):
    with torch.no_grad():
        sp = net.edge_params.sign.raw_values
        v = sp.data.cpu().numpy().copy()
        n_exc, n_inh = int((v > 0).sum()), int((v < 0).sum())
        sp.data = torch.tensor(
            rng.permutation(np.array([1.0]*n_exc + [-1.0]*n_inh, dtype=np.float32)),
            dtype=torch.float32).to(DEVICE)
        raw = net.edge_params.syn_strength.raw_values
        s = raw.data.cpu().numpy().copy()
        pos, neg = np.where(s > 0)[0], np.where(s <= 0)[0]
        if len(pos) > 1: s[pos] = rng.permutation(s[pos])
        if len(neg) > 1: s[neg] = rng.permutation(s[neg])
        raw.data = torch.tensor(s, dtype=torch.float32).to(DEVICE)
    return net


def is_stable(network, stimulus, dt):
    """Dynamic stability check for a single forward pass.

    Deliberately does NOT wrap the forward pass in try/except. The original
    implementation caught all exceptions and returned False, conflating "this
    network is dynamically unstable" with "something in the code or environment
    broke." A CUDA OOM, a shape error, or an op lacking a deterministic
    implementation would each have been counted silently as instability.

    The original reported zero rejections across 150 models. Had any of them
    thrown, that fact would have been invisible. (Nothing did throw. The code
    could not have told us.)

    Genuine instability produces non-finite activations or activations exceeding
    OVERFLOW_LIMIT; both are caught by the explicit checks below. Anything else
    is a bug, and a bug should raise.
    """
    stim = stimulus.unsqueeze(1) if stimulus.dim() == 2 else stimulus
    s0 = network.fade_in_state(1.0, dt, stim[[0]])
    out = network.simulate(stim[None], dt, initial_state=s0)
    o = out.cpu().numpy()
    return bool(torch.all(torch.isfinite(out)) and np.all(np.abs(o) < OVERFLOW_LIMIT))


# ── MAIN LOOP (rejection counting fixed) ──────────────────────────────────────
def run_condition(label, make_fn, n_models, dataset, stim_indices, dt):
    print(f"\n{'='*60}\nCondition: {label} (n={n_models})\n{'='*60}")

    check = dataset[stim_indices[0]]
    if not isinstance(check, torch.Tensor):
        check = torch.tensor(check, dtype=torch.float32)
    check = check.to(DEVICE)

    pop_matrices, rdms, rdms_euc = [], [], []
    rejections_before_accept = []   # FIXED: was attempts_log.append(seed)
    accepted, seed, cell_types = 0, 0, None
    rejects_this_model = 0

    while accepted < n_models:
        if seed > n_models * MAX_ATTEMPTS:
            print(f"  WARNING: gave up after {seed} seeds."); break
        net = make_fn(seed); seed += 1

        if not is_stable(net, check, dt):
            rejects_this_model += 1
            del net; torch.cuda.empty_cache(); continue

        if cell_types is None:
            cell_types = get_cell_types(net)

        pop_vecs, ok = [], True
        for si in stim_indices:
            s = dataset[si]
            if not isinstance(s, torch.Tensor):
                s = torch.tensor(s, dtype=torch.float32)
            pv = get_population_vector(net, s.to(DEVICE), dt, cell_types)
            if not np.isfinite(pv).all(): ok = False; break
            pop_vecs.append(pv)
        if not ok:
            rejects_this_model += 1
            del net; torch.cuda.empty_cache(); continue

        pm = np.stack(pop_vecs, axis=0)          # (12, 65) float64
        pop_matrices.append(pm)
        rdms.append(build_rdm(pm, metric="cosine"))
        rdms_euc.append(build_rdm(pm, metric="euclidean_normalized"))
        rejections_before_accept.append(rejects_this_model)
        rejects_this_model = 0
        accepted += 1
        print(f"  Accepted {accepted}/{n_models} (seed={seed-1})", flush=True)
        del net; torch.cuda.empty_cache()

    pop_matrices = np.stack(pop_matrices, 0)
    rdms         = np.stack(rdms, 0)
    rdms_euc     = np.stack(rdms_euc, 0)

    rej = np.array(rejections_before_accept)
    print(f"\n  {label}: {accepted}/{n_models} accepted over {seed} seeds.")
    print(f"  Rejections before acceptance: total {rej.sum()}, "
          f"mean {rej.mean():.2f}, first-try {int((rej == 0).sum())}/{accepted}")
    print("  (The original reported 'mean attempts 25.5 +/- 14.4', which was the "
          "mean of the\n   cumulative seed counter 1..50 -- a loop index, not a "
          "resampling statistic.)")
    return pop_matrices, rdms, rdms_euc, accepted, seed, cell_types


# ── BIOLOGICAL REFERENCE ──────────────────────────────────────────────────────
def build_bio_rdm():
    """Von Mises T4/T5 reference. NOTE: this reference is itself ~0.978 circular
    (Spearman, against min(|i-j|, 12-|i-j|)), so a raw correlation against it
    largely measures a network's circular organization rather than its
    biological fidelity. The partial correlation is the interpretable quantity.
    """
    dirs = np.linspace(0, 330, N_DIR)
    prefs = [180, 0, 90, 270, 180, 0, 90, 270]
    kappa = 2.5
    def vm(t, mu):
        r = np.exp(kappa * (np.cos(np.radians(t - mu)) - 1.0))
        r = r - np.exp(-2.0 * kappa)
        return max(r, 0.0)
    pop = np.array([[vm(d, mu) for mu in prefs] for d in dirs], dtype=np.float64)
    pop = pop / pop.max(axis=0, keepdims=True)
    return build_rdm(pop, metric="cosine")


# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    dataset = MovingEdge(offsets=[-10, 11], intensities=[0, 1], speeds=[19],
                         height=80, post_pad_mode="continue",
                         t_pre=1.0, t_post=1.0, dt=1/200, angles=ANGLES)
    on_idx = [i for i, r in dataset.arg_df.iterrows() if r["intensity"] == INTENSITY]
    assert len(on_idx) == N_DIR

    cc_pop,  cc_rdms,  cc_euc,  cc_n,  _, cell_types = run_condition(
        "Untrained CC", make_untrained_cc, N_MODELS, dataset, on_idx, dataset.dt)
    syn_pop, syn_rdms, syn_euc, syn_n, _, _ = run_condition(
        "Untrained Random (syn shuffle)",
        lambda s: apply_syn_shuffle(make_untrained_cc(s),
                                    np.random.default_rng(s + 10_000)),
        N_MODELS, dataset, on_idx, dataset.dt)
    sign_pop, sign_rdms, sign_euc, sign_n, _, _ = run_condition(
        "Untrained Random (sign shuffle)",
        lambda s: apply_sign_shuffle(make_untrained_cc(s),
                                     np.random.default_rng(s + 20_000)),
        N_MODELS, dataset, on_idx, dataset.dt)

    cc_mean   = cc_rdms.mean(0)
    syn_mean  = syn_rdms.mean(0)
    sign_mean = sign_rdms.mean(0)

    cc_mean_euc   = cc_euc.mean(0)
    syn_mean_euc  = syn_euc.mean(0)
    sign_mean_euc = sign_euc.mean(0)

    # -- CANCELLATION-FREE CROSS-CHECK. Does the metric change the answer? ----
    # Euclidean distance on L2-normalized vectors is monotone in the angle and
    # involves no subtraction from 1, so it cannot suffer the cancellation that
    # 1 - cos(theta) can. If the cosine and euclidean RDMs induce the same rank
    # order, the cosine RDM carries real signal. If they disagree, the rank
    # order is arbitrary. This test does not depend on RESOLVE_MARGIN.
    print("\n" + "=" * 70)
    print("CROSS-CHECK -- cosine vs euclidean-normalized (cancellation-free)")
    print("=" * 70)
    print("  Kendall tau between the two metrics' rank orders.")
    print("  tau ~ 1.0 => metric is irrelevant => real signal.")
    print("  tau ~ 0.0 => rank order is arbitrary => noise.\n")
    for name, R_cos, R_euc in [("CC", cc_mean, cc_mean_euc),
                               ("Rand-syn", syn_mean, syn_mean_euc),
                               ("Rand-sign", sign_mean, sign_mean_euc)]:
        t = kendalltau(upper(R_cos), upper(R_euc)).statistic
        print(f"   {name:10s} mean-RDM tau(cosine, euclidean) = {t:+.4f}")

    # Per-model is the stricter test: a mean can look coherent even when every
    # individual RDM that went into it is noise.
    print()
    for name, Rc, Re in [("CC", cc_rdms, cc_euc),
                         ("Rand-syn", syn_rdms, syn_euc),
                         ("Rand-sign", sign_rdms, sign_euc)]:
        taus = np.array([kendalltau(upper(Rc[i]), upper(Re[i])).statistic
                         for i in range(Rc.shape[0])])
        print(f"   {name:10s} per-model tau: median {np.median(taus):+.4f}, "
              f"min {taus.min():+.4f}, frac > 0.9: {(taus > 0.9).mean():.2f}")

    # ── THE GUARD. Everything downstream is invalid if this fails. ────────────
    print("\n" + "="*70)
    print("PRECISION GUARD -- can these RDMs be resolved above float32 round-off?")
    print("="*70)
    assert_rdm_resolvable(cc_mean,   cc_pop.reshape(-1, cc_pop.shape[-1]),   "CC")
    assert_rdm_resolvable(syn_mean,  syn_pop.reshape(-1, syn_pop.shape[-1]), "Rand-syn")
    assert_rdm_resolvable(sign_mean, sign_pop.reshape(-1, sign_pop.shape[-1]), "Rand-sign")
    print("\nAll RDMs resolvable. Proceeding.\n")

    # ── RSA, with BOTH p-values ──────────────────────────────────────────────
    circ = circular_reference()
    bio  = build_bio_rdm()
    print(f"r(biological reference, circular) = "
          f"{spearmanr(upper(bio), upper(circ)).statistic:.4f}   "
          f"<- the reference is nearly a circular-distance matrix\n")

    for name, other in [("Rand-syn", syn_mean), ("Rand-sign", sign_mean)]:
        rs, ps, rk, pk = rdm_similarity(cc_mean, other)
        r_o, p_perm, t_o, pt_perm, _, _ = permutation_test_rdm(cc_mean, other)
        print(f"CC vs {name}:")
        print(f"   Spearman r = {rs:+.3f}   p_analytical = {ps:.4f}   "
              f"p_perm = {p_perm:.4f}")
        print(f"   Kendall  t = {rk:+.3f}   p_analytical = {pk:.4f}   "
              f"p_perm = {pt_perm:.4f}")

    print("\nCircularity of each condition (r against circular reference):")
    for name, R in [("CC", cc_mean), ("Rand-syn", syn_mean), ("Rand-sign", sign_mean)]:
        r_c = spearmanr(upper(R), upper(circ)).statistic
        r_b = spearmanr(upper(R), upper(bio)).statistic
        pr  = partial_spearman(upper(R), upper(bio), upper(circ))
        print(f"   {name:10s} r(circ)={r_c:+.3f}   r(bio)={r_b:+.3f}   "
              f"r(bio|circ)={pr:+.3f}")
    print("\n  If r(bio) tracks r(circ) for every condition, the biological "
          "comparison is\n  measuring circular organization, not biological "
          "fidelity. Report the partial.")

    # within-CC consistency
    cons = [spearmanr(upper(cc_rdms[i]), upper(cc_rdms[j])).statistic
            for i, j in combinations(range(len(cc_rdms)), 2)]
    print(f"\nWithin-CC consistency: r = {np.mean(cons):.3f} +/- {np.std(cons):.3f}")

    np.savez(results_dir / "results_exp4_untrained_corrected.npz",
             cc_pop_matrices=cc_pop, syn_pop_matrices=syn_pop,
             sign_pop_matrices=sign_pop,
             cc_rdms=cc_rdms, syn_rdms=syn_rdms, sign_rdms=sign_rdms,
             cc_rdms_euc=cc_euc, syn_rdms_euc=syn_euc, sign_rdms_euc=sign_euc,
             cc_mean_rdm=cc_mean, syn_mean_rdm=syn_mean, sign_mean_rdm=sign_mean,
             cc_mean_rdm_euc=cc_mean_euc, syn_mean_rdm_euc=syn_mean_euc,
             sign_mean_rdm_euc=sign_mean_euc,
             bio_rdm=bio, circular_reference=circ, cell_types=np.array(cell_types))
    print("\nSaved results/results_exp4_untrained_corrected.npz")

Device: cuda


building stimuli:   0%|          | 0/24 [00:00<?, ?it/s]


Condition: Untrained CC (n=50)
  Accepted 1/50 (seed=0)
  Accepted 2/50 (seed=1)
  Accepted 3/50 (seed=2)
  Accepted 4/50 (seed=3)
  Accepted 5/50 (seed=4)
  Accepted 6/50 (seed=5)
  Accepted 7/50 (seed=6)
  Accepted 8/50 (seed=7)
  Accepted 9/50 (seed=8)
  Accepted 10/50 (seed=9)
  Accepted 11/50 (seed=10)
  Accepted 12/50 (seed=11)
  Accepted 13/50 (seed=12)
  Accepted 14/50 (seed=13)
  Accepted 15/50 (seed=14)
  Accepted 16/50 (seed=15)
  Accepted 17/50 (seed=16)
  Accepted 18/50 (seed=17)
  Accepted 19/50 (seed=18)
  Accepted 20/50 (seed=19)
  Accepted 21/50 (seed=20)
  Accepted 22/50 (seed=21)
  Accepted 23/50 (seed=22)
  Accepted 24/50 (seed=23)
  Accepted 25/50 (seed=24)
  Accepted 26/50 (seed=25)
  Accepted 27/50 (seed=26)
  Accepted 28/50 (seed=27)
  Accepted 29/50 (seed=28)
  Accepted 30/50 (seed=29)
  Accepted 31/50 (seed=30)
  Accepted 32/50 (seed=31)
  Accepted 33/50 (seed=32)
  Accepted 34/50 (seed=33)
  Accepted 35/50 (seed=34)
  Accepted 36/50 (seed=35)
  Accepted 37/5

SystemExit: 
ABORT: the CC RDM's dynamic range does not exceed the float32 round-off floor of the responses it was computed from.
Its rank order -- and therefore every Spearman/Kendall statistic and permutation test derived from it -- would be numerical noise.

This is exactly the failure mode of the original Experiment 4, whose RDM entries (7e-9 .. 5e-8) sat below a ~6e-8 floor.

Remedies (each changes what the experiment measures; choose deliberately):
  (a) raise BIAS_NOISE / SYN_STRENGTH_NOISE until untrained responses actually differ across directions, and report the sensitivity analysis;
  (b) use metric='euclidean_normalized', which is cancellation-free;
  (c) conclude that untrained networks have no measurable representational geometry at this perturbation scale -- which would mean training CREATES the directional structure rather than amplifying a wiring prior.


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Download Files

In [ ]:
# Download results file and figures to local machine
from google.colab import files
files.download("results/results_exp4_untrained.npz")
files.download("figures/exp4_untrained_rdms.png")
files.download("figures/exp4_untrained_permtest.png")

## Results

*All results in this section are from `results_exp4_untrained_corrected.npz`,
N = 50 models per condition, at the Flyvis prior perturbation
(BIAS\_NOISE = 0.05, SYN\_STRENGTH\_NOISE = 0.002).*

### The experiment terminates at the precision guard

This experiment does not report RSA statistics. It aborts before computing them,
because the RDMs it would compute them from carry no rank order.

**Guard verdict, untrained CC:**

| quantity | value |
|---|---|
| RDM off-diagonal range | 4.96×10⁻¹⁰ … 1.71×10⁻⁸ |
| RDM off-diagonal span | 1.66×10⁻⁸ |
| response magnitude, max\|v\| | 1.615 |
| float32 round-off floor (ε\_f32 × max\|v\|) | 1.93×10⁻⁷ |
| **span / floor** | **0.09×** (threshold: ≥ 10×) |
| resolvable | **False** |

The float32 round-off floor of the responses is eleven times larger than the entire
dynamic range of the RDM derived from them. Population vectors are float32 with
magnitude ≈ 0.5; for the near-identical responses untrained networks produce across
the twelve directions, the quotient in `1 − dot/(‖a‖‖b‖)` lies within one ULP of 1.0
and the subtraction retains no significant digits. The resulting entries are not
small-but-structured. They are rounding.

### The metric cross-check confirms this is not an artifact of cosine distance

Each population matrix is used to build two RDMs: cosine (chord form, float64) and
Euclidean-normalized (monotone in the angle, no subtraction from 1, therefore
cancellation-free). If the two induce the same rank order, the metric is irrelevant
and whatever ordering exists is real.

| condition | mean-RDM τ | per-model τ: median | min | frac > 0.9 |
|---|---|---|---|---|
| CC | +0.945 | **+1.0000** | +1.0000 | 1.00 |
| Rand-syn | +0.853 | **+1.0000** | +1.0000 | 1.00 |
| Rand-sign | +0.751 | **+1.0000** | +1.0000 | 1.00 |

**Per-model Kendall τ is exactly 1.0000 for all 150 models.** The two metrics agree
perfectly on every individual RDM, and the guard establishes that what they agree on
is noise. The failure is in the resolution of the responses, not the algebra of the
distance function. This test does not depend on the guard's threshold, which is a
heuristic.

A second observation, incidental but recorded: mean-RDM τ falls to 0.945, 0.853,
0.751 while per-model τ stays at 1.0. Averaging fifty RDMs breaks the metric
agreement, worst for the condition with the largest responses. The mean of a set of
distance matrices is not itself a distance matrix, and its rank order depends on
which distance was averaged. This is separate from the resolution floor and is not
fixed by fixing the floor.

### Stability and acceptance

**0 rejections; 50/50 first-try acceptance in all three conditions.** Untrained
networks at the Flyvis initialization are uniformly dynamically stable; the stability
constraint imposed no filtering. This contrasts with the trained-network baselines of
Experiments 1–2, where the full Shiu-style shuffle required resampling. Dynamic
instability is a property of the trained parameter regime, not of the architecture at
initialization.

This result is unaffected by the numerical problem above. It depends only on whether
activations remained finite and bounded, which is a robust check.

*Two corrections.* The original reported "mean attempts 25.5 ± 14.4" in support of the
acceptance claim. That statistic was the mean and standard deviation of the cumulative
seed counter 1…50 — a loop index, not a resampling statistic. The claim was true; the
number cited for it carried no information. And the original `is_stable` wrapped the
forward pass in `try/except Exception: return False`, which would have counted a CUDA
error, a shape mismatch, or a non-deterministic-op failure as dynamic instability,
silently. Nothing threw in practice; the code could not have told us.

## Retraction

The following were reported in an earlier version of this experiment and in the
preprint. **All are permutation tests on floating-point rounding. None should be
cited.**

| reported quantity | reported value | status |
|---|---|---|
| CC vs Rand-syn | r = 0.260, p\_perm = 0.041 | void |
| CC vs Rand-sign | r = 0.215, p\_perm = 0.048 | void |
| CC vs Biology | r = −0.015, p\_perm = 0.561 | void |
| Rand-syn vs Biology | r = 0.471, p\_perm \< 0.0001 | void |
| Rand-sign vs Biology | r = 0.585, p\_perm \< 0.0001 | void |
| Within-CC consistency | r = 0.006 ± 0.133 | void |

Three claims are withdrawn.

**"The mean untrained CC cosine RDM reveals a clear block-diagonal / anti-diagonal
structure… this circular direction gradient is present before any gradient-based
optimization."** The stored CC mean RDM correlates with a circular-distance reference
at −0.046. Recomputing the identical mean RDM through a float64 path gives +0.463; a
second float64 path gives +0.175. The three disagree because there is no correct value
to recover. There is no circular direction gradient, and no way to determine from these
responses whether one exists.

**"What matters is the relative geometry within each RDM, not the absolute scale."** At
10⁻⁸, computed from float32 responses, there is no relative geometry. The scale is not
a nuisance to normalize away; it is the reason the matrix is empty.

**"Progressive degradation from CC through syn-shuffle to sign-shuffle."** The three
conditions' RDM spans are 1.03×10⁻⁸, 2.47×10⁻⁷, and 3.50×10⁻⁶ — a 340-fold range. The
claimed progression is that conditioning gradient, reversed. Rand-sign was not less
degraded; it was less ill-conditioned, because scrambling E/I balance inflates responses
tenfold. (Spans quoted here are from the N=5 sweep; see Experiment 4b.)

Two candidate explanations were checked and excluded. The `+1e-10` epsilon in the
original `build_rdm` is inert: `float32(0.53) + 1e-10 == float32(0.53)` exactly, the ULP
at 0.53 being ≈ 6×10⁻⁸. And the algebraic form of cosine distance is not at fault: given
float64 inputs, `scipy.spatial.distance.cosine` agrees with the chord identity to six
significant figures. **The dtype of the input — not the formula, not the epsilon —
destroys the result.**

## Discussion

- **The untrained CC RDM is unresolvable at the Flyvis prior.** Its dynamic range is an
  order of magnitude below the numerical resolution of the responses it derives from. No
  Spearman, Kendall, or permutation statistic computed from it carries information. This
  is a measurement, not a modeling failure.

- **The problem is the dtype, and it is confirmed independently of any threshold.**
  Per-model Kendall τ between the cosine and Euclidean-normalized rank orders is 1.0000
  for all 150 models. Both metrics see the same thing. The guard's 10× margin is a
  heuristic; this cross-check is not.

- **Instability is training-dependent.** 100% first-try acceptance across all three
  untrained conditions confirms that the dynamic instability documented in Experiments
  1–2 is a property of the trained parameter regime, not the architecture.

- **Whether untrained networks have geometry at *any* perturbation scale is not answered
  here.** The original's limitations section proposed larger perturbations as the remedy.
  That analysis is Experiment 4b.

- **The methodological finding may outlast the empirical one.** The original reported two
  headline p-values of 0.041 and 0.048 — sitting just under threshold, as random rank
  vectors do. Nothing in the analysis pipeline asked whether the matrices being tested
  carried a rank order. The corrected code refuses to compute a statistic on an RDM whose
  dynamic range does not exceed the round-off of its own inputs.